In [166]:
import json 
from collections import Counter
import seaborn as sns
import pandas as pd
from pathlib import Path
import pickle
from networkx.algorithms import community
import networkx as nx
from operator import itemgetter
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
plt.tight_layout()

<Figure size 432x288 with 0 Axes>

In [77]:
ig_path = "../models/gcn/ig_attrs.json"
shap_path = "../models/gcn/shap_values.json"

ig_df = pd.read_json(ig_path).T
ig_df

,label,rel_doc_ids,rel_doc_labels
27456,Blutdrucksenker_beides,"[27995, 897, 27456, 896, 899, 903, 901, 902, 9...","[Blutdrucksenker_Blutdruck, Cholesterinsenker_..."
27457,Blutdrucksenker_Herzschw,"[27995, 27457, 897, 896, 899, 903, 901, 902, 9...","[Blutdrucksenker_Blutdruck, Blutdrucksenker_He..."
27458,Blutdrucksenker_Blutdruck,"[1048, 27458, 1053, 1761, 20, 1211, 1016, 1728...","[Blutdrucksenker_Blutdruck, Blutdrucksenker_Bl..."
27459,Cholesterinsenker_unklar,"[27995, 899, 900, 898, 905, 903, 904, 902, 901...","[Blutdrucksenker_Blutdruck, Cholesterinsenker_..."
27460,Blutdrucksenker_unklar,"[27460, 897, 898, 896, 903, 901, 902, 900, 899...","[Blutdrucksenker_unklar, Cholesterinsenker_bei..."
...,...,...,...
27991,Blutdrucksenker_Blutdruck,"[27995, 899, 900, 898, 27991, 897, 903, 904, 9...","[Blutdrucksenker_Blutdruck, Cholesterinsenker_..."
27992,Blutdrucksenker_beides,"[27995, 899, 900, 27992, 898, 897, 903, 904, 9...","[Blutdrucksenker_Blutdruck, Cholesterinsenker_..."
27993,Cholesterinsenker_beides,"[27995, 899, 27993, 900, 898, 897, 903, 904, 9...","[Blutdrucksenker_Blutdruck, Cholesterinsenker_..."
27994,Blutdrucksenker_Blutdruck,"[27995, 27994, 899, 900, 898, 897, 903, 904, 9...","[Blutdrucksenker_Blutdruck, Blutdrucksenker_Bl..."


In [78]:
ig_df['rel'] = list(zip(ig_df.rel_doc_ids, ig_df.rel_doc_labels))
ig_df['rel'] = ig_df['rel'].apply(lambda x: list(zip(*x)))
ig_df = ig_df.explode("rel").drop(columns=['rel_doc_ids', 'rel_doc_labels'])
ig_df

,label,rel
27456,Blutdrucksenker_beides,"(27995, Blutdrucksenker_Blutdruck)"
27456,Blutdrucksenker_beides,"(897, Cholesterinsenker_beides)"
27456,Blutdrucksenker_beides,"(27456, Blutdrucksenker_beides)"
27456,Blutdrucksenker_beides,"(896, DM_nur Tabletten)"
27456,Blutdrucksenker_beides,"(899, Cholesterinsenker_beides)"
...,...,...
27995,Blutdrucksenker_Blutdruck,"(903, Blutdrucksenker_beides)"
27995,Blutdrucksenker_Blutdruck,"(904, Cholesterinsenker_Hypercholesterin)"
27995,Blutdrucksenker_Blutdruck,"(902, DM_Insulin und Tabletten)"
27995,Blutdrucksenker_Blutdruck,"(901, Blutdrucksenker_beides)"


In [81]:
ig_df[['rel_id','rel_type']] = ig_df['rel'].apply(pd.Series)
ig_df = ig_df.drop(columns=['rel']).reset_index()
ig_df

,index,label,rel_id,rel_type
0,27456,Blutdrucksenker_beides,27995,Blutdrucksenker_Blutdruck
1,27456,Blutdrucksenker_beides,897,Cholesterinsenker_beides
2,27456,Blutdrucksenker_beides,27456,Blutdrucksenker_beides
3,27456,Blutdrucksenker_beides,896,DM_nur Tabletten
4,27456,Blutdrucksenker_beides,899,Cholesterinsenker_beides
...,...,...,...,...
5395,27995,Blutdrucksenker_Blutdruck,903,Blutdrucksenker_beides
5396,27995,Blutdrucksenker_Blutdruck,904,Cholesterinsenker_Hypercholesterin
5397,27995,Blutdrucksenker_Blutdruck,902,DM_Insulin und Tabletten
5398,27995,Blutdrucksenker_Blutdruck,901,Blutdrucksenker_beides


In [89]:
G=nx.from_pandas_edgelist(ig_df, "index", 'rel_id')

In [140]:
id_df = ig_df.iloc[:,0:2]
label_df = ig_df.iloc[:,2:4]

new_columns = ["id", "label"]
id_df.columns = new_columns
label_df.columns = new_columns

id_label_df = pd.concat([id_df, label_df], ignore_index=True).drop_duplicates()
id2label = dict(zip(id_label_df.id, id_label_df.label))
nx.set_node_attributes(G, id2label, "label")

In [146]:
nx.is_connected(G)

False

In [152]:
components = nx.connected_components(G)
largest_component = max(components, key=len)
subgraph = G.subgraph(largest_component)
diameter = nx.diameter(subgraph)
print("Network diameter of largest component:", diameter)

Network diameter of largest component: 20


In [154]:
triadic_closure = nx.transitivity(G)
print("Triadic closure:", triadic_closure)

Triadic closure: 0.009142805854088406


In [161]:
degree_dict = dict(G.degree(G.nodes()))
nx.set_node_attributes(G, degree_dict, 'degree')
sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)
print(sorted_degree[:30])

[(901, 492), (899, 491), (903, 491), (902, 491), (900, 491), (904, 463), (898, 455), (27995, 407), (897, 266), (896, 205), (905, 138), (895, 44), (27458, 15), (27749, 14), (27969, 13), (27472, 12), (27947, 12), (27621, 12), (27521, 12), (27919, 12), (27560, 12), (27567, 12), (27737, 12), (27456, 11), (27457, 11), (27460, 11), (27461, 11), (27462, 11), (27463, 11), (27464, 11)]


In [163]:
betweenness_dict = nx.betweenness_centrality(G) # Run betweenness centrality
nx.set_node_attributes(G, betweenness_dict, 'betweenness')
sorted_betweenness = sorted(betweenness_dict.items(), key=itemgetter(1), reverse=True)
print(sorted_betweenness[:20])

[(27969, 0.2845631774735224), (905, 0.27469263238739405), (27915, 0.2637888454414488), (27775, 0.25481637739818436), (1018, 0.2526190966199198), (27995, 0.17670731174473808), (27607, 0.16901970021733043), (27749, 0.1280791214750283), (27932, 0.10372137124040806), (959, 0.09384192038618251), (1294, 0.08079876321798524), (901, 0.06890775549623511), (27472, 0.06423524338733227), (898, 0.062336280134421856), (906, 0.06061957843451293), (1839, 0.06011467837460829), (27947, 0.04871176706945383), (899, 0.04739230561119185), (903, 0.04739230561119185), (902, 0.04739230561119185)]


In [164]:
top_betweenness = sorted_betweenness[:20]
for tb in top_betweenness:
    degree = degree_dict[tb[0]] # Use degree_dict to access a node's degree, see footnote 2
    print("Name:", tb[0], "| Betweenness Centrality:", tb[1], "| Degree:", degree)

Name: 27969 | Betweenness Centrality: 0.2845631774735224 | Degree: 13
Name: 905 | Betweenness Centrality: 0.27469263238739405 | Degree: 138
Name: 27915 | Betweenness Centrality: 0.2637888454414488 | Degree: 11
Name: 27775 | Betweenness Centrality: 0.25481637739818436 | Degree: 11
Name: 1018 | Betweenness Centrality: 0.2526190966199198 | Degree: 2
Name: 27995 | Betweenness Centrality: 0.17670731174473808 | Degree: 407
Name: 27607 | Betweenness Centrality: 0.16901970021733043 | Degree: 11
Name: 27749 | Betweenness Centrality: 0.1280791214750283 | Degree: 14
Name: 27932 | Betweenness Centrality: 0.10372137124040806 | Degree: 11
Name: 959 | Betweenness Centrality: 0.09384192038618251 | Degree: 3
Name: 1294 | Betweenness Centrality: 0.08079876321798524 | Degree: 4
Name: 901 | Betweenness Centrality: 0.06890775549623511 | Degree: 492
Name: 27472 | Betweenness Centrality: 0.06423524338733227 | Degree: 12
Name: 898 | Betweenness Centrality: 0.062336280134421856 | Degree: 455
Name: 906 | Betwee

In [167]:
communities = community.greedy_modularity_communities(G)

In [169]:
modularity_dict = {}
for i,c in enumerate(communities):
    for name in c:
        modularity_dict[name] = i
nx.set_node_attributes(G, modularity_dict, 'modularity')

In [171]:
nx.write_gexf(G, 'ig_graph_1.0.gexf')